# Window Soft Argmax Inference with Pretrained Models - AP-10K

This notebook performs semantic correspondence inference using the Window Soft Argmax (WSA) strategy on AP-10K animal pose dataset with pretrained models.

## What this notebook does:
- Tests all backbone models (DINOv2, DINOv3, SAM) with pretrained weights on AP-10K
- Uses Window Soft Argmax prediction strategy for more precise localization
- Evaluates on AP-10K test split (only split available)
- Reports PCK@α metrics for α ∈ {0.05, 0.1, 0.2}
- Handles variable keypoint pairs across different animal species

## WSA Strategy:
- Creates a window around the argmax peak
- Applies softmax with temperature scaling
- Computes weighted average of coordinates within the window
- Generally provides better localization than simple argmax

## AP-10K Dataset Specifics:
- Animal pose dataset with 17 keypoints per animal
- Different species have different visible keypoints
- Automatic skip logic for pairs with no common keypoints
- Uses COCO format annotations

## Configuration options:
- Change `BACKBONE` to 'dinov2', 'dinov3', or 'sam'
- AP-10K only has 'test' split available

## Expected runtime:
- ~10-20 minutes per backbone (depends on number of valid pairs)

In [ ]:
%pip install torchmetrics                # ONLY FOR FIRST EXECUTION
%pip install git+https://github.com/facebookresearch/segment-anything.git

from google.colab import drive
import os

REPO_URL = "https://github.com/AML-Semantic-Correspondence/Semantic_Correspondence.git"

# 2. Clone/Pull the Code (access to logic)
print("\n Setting up repository...")

# First ensure we're in a safe directory
%cd /content

# Clean up any existing problematic directories
if os.path.exists('/content/Semantic_Correspondence'):
    print("Removing existing Semantic_Correspondence directory...")
    !rm -rf /content/Semantic_Correspondence

if os.path.exists('/content/semantic-correspondence'):
    print("Removing existing semantic-correspondence directory...")
    !rm -rf /content/semantic-correspondence

# Clone the repository
print("Cloning repository fresh...")
try:
    !git clone {REPO_URL}
    
    # The repo will be cloned as 'Semantic_Correspondence', let's rename it for consistency
    if os.path.exists('/content/Semantic_Correspondence'):
        !mv /content/Semantic_Correspondence /content/semantic-correspondence
        print("Repository cloned and renamed successfully")
    else:
        print("Repository clone failed")
except Exception as e:
    print(f"Error during clone: {e}")

# Mount drive and extract AP-10K dataset
drive.mount("/content/drive", force_remount=True)

# Extract AP-10K dataset
print("Extracting AP-10K dataset...")
try:
    !unzip -o -q "/content/drive/MyDrive/AML-Semantic-Correspondence/datasets/ap-10k.zip"
    print("AP-10K dataset extracted successfully")
except Exception as e:
    print(f"Error extracting AP-10K dataset: {e}")

# Add the repository to path
%cd /content/semantic-correspondence
import sys
sys.path.append('/content/semantic-correspondence')

# Import inference functions
from src.inference.run_evaluation import run_evaluation
from src.inference.wsa import wsa_strategy

# ===== CONFIGURATION =====
# Change these parameters to test different combinations
BACKBONE = 'dinov2'  # Options: 'dinov2', 'dinov3', 'sam'
SPLIT = 'test'       # AP-10K only has test split

print(f"\nRunning WSA Inference (Pretrained) - AP-10K")
print(f"Configuration:")
print(f"   - Backbone: {BACKBONE}")
print(f"   - Dataset: ap-10k")
print(f"   - Split: {SPLIT}")
print(f"   - Strategy: Window Soft Argmax (WSA)")
print(f"   - Model: Pretrained (no fine-tuning)")

# Run evaluation
print("\nStarting evaluation...")
run_evaluation(
    backbone=BACKBONE,
    dataset_var='ap-10k',
    split=SPLIT,
    prediction_method=wsa_strategy,
    weights_path=None  # Use pretrained weights
)

print("\nEvaluation completed!")
print("\nTo test other backbones, modify the BACKBONE variable above and re-run the cell.")
print("\nNote: WSA typically performs better than simple argmax, especially for precise localization tasks.")
print("AP-10K automatically skips pairs with no common keypoints between different animal species.")